In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

In [2]:
pio.templates.default = "plotly_white"
DATA_DIR = Path("../data")
SYMBOLS = ["SBIN", "LICI", "HAL", "NTPC", "ONGC"]
START, END = "2025-01-01", "2026-01-31"

In [3]:
stocks = {}
quality = []
for symbol in SYMBOLS:
    files = sorted((DATA_DIR / symbol).glob("*/*.csv"))
    
    if not files:
        raise FileNotFoundError(f"No monthly CSVs in {DATA_DIR / symbol}")
    
    df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
    
    df["date"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.normalize()
    
    df = df.loc[df["date"].between(START, END), ["date", "open", "high", "low", "close", "volume"]].copy()
    
    duplicates = int(df["date"].duplicated().sum())
    
    df = df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
    
    for column in ["open", "high", "low", "close", "volume"]:
        df[column] = pd.to_numeric(df[column], errors="raise")
    
    if df.empty or not np.isfinite(df[["open", "high", "low", "close", "volume"]].to_numpy()).all():
        raise ValueError(f"{symbol}: empty dataset or missing/non-finite OHLCV")
    
    bad = (df[["open", "high", "low", "close"]].le(0).any(axis=1)
           | df["volume"].lt(0)
           | df["high"].lt(df[["open", "low", "close"]].max(axis=1))
           | df["low"].gt(df[["open", "high", "close"]].min(axis=1)))
    
    if bad.any():
        raise ValueError(f"{symbol}: {int(bad.sum())} invalid OHLCV rows; inspect your CSVs")
    
    for days in [10, 20, 50]:
        df[f"MA{days}"] = df["close"].rolling(days).mean()
    
    df["return_pct"] = df["close"].pct_change(fill_method=None) * 100
    
    stocks[symbol] = df
    
    quality.append({
        "symbol":symbol,
        "rows":len(df),
        "logged_rows":269,
        "matches_logged_total":len(df) == 269,
        "duplicate_dates_removed":duplicates,
        "first_date":df["date"].min(),
        "last_date":df["date"].max()
    })

display(pd.DataFrame(quality))

,symbol,rows,logged_rows,matches_logged_total,duplicate_dates_removed,first_date,last_date
0,SBIN,269,269,True,0,2025-01-01,2026-01-30
1,LICI,269,269,True,0,2025-01-01,2026-01-30
2,HAL,269,269,True,0,2025-01-01,2026-01-30
3,NTPC,269,269,True,0,2025-01-01,2026-01-30
4,ONGC,269,269,True,0,2025-01-01,2026-01-30


In [4]:
logged_counts = [23, 20, 19, 19, 21, 21, 23, 19, 22, 21, 19, 22, 20]
logged_months = pd.date_range("2025-01-01", "2026-01-01", freq="MS").strftime("%Y-%m")

expected = pd.Series(logged_counts, index=logged_months)

coverage = []
for symbol, df in stocks.items():
    actual = df.groupby(df["date"].dt.strftime("%Y-%m")).size().reindex(logged_months, fill_value=0)
    
    coverage.append(pd.DataFrame({
        "symbol":symbol, 
        "month":logged_months,
        "logged":expected.to_numpy(),
        "actual":actual.to_numpy()
    }))

coverage = pd.concat(coverage, ignore_index=True)
coverage["matches"] = coverage["logged"] == coverage["actual"]

display(coverage)

,symbol,month,logged,actual,matches
0,SBIN,2025-01,23,23,True
1,SBIN,2025-02,20,20,True
2,SBIN,2025-03,19,19,True
3,SBIN,2025-04,19,19,True
4,SBIN,2025-05,21,21,True
...,...,...,...,...,...
60,ONGC,2025-09,22,22,True
61,ONGC,2025-10,21,21,True
62,ONGC,2025-11,19,19,True
63,ONGC,2025-12,22,22,True


In [5]:
summary = []
for symbol, df in stocks.items():
    summary.append({
        "symbol":symbol, 
        "min_close":df["close"].min(), 
        "max_close":df["close"].max(),
        "mean_close":df["close"].mean(), 
        "std_close":df["close"].std(),
        "mean_volume":df["volume"].mean(), 
        "mean_daily_return_pct":df["return_pct"].mean(),
        "daily_volatility_pct":df["return_pct"].std()
    })

summary = pd.DataFrame(summary)

display(summary.round(3))

,symbol,min_close,max_close,mean_close,std_close,mean_volume,mean_daily_return_pct,daily_volatility_pct
0,SBIN,688.80,1077.15,841.209,91.457,1.047873e+07,0.121,1.135
1,LICI,364.60,486.40,431.056,28.529,2.410148e+06,-0.022,1.383
2,HAL,3088.20,5127.70,4421.419,464.409,1.601777e+06,0.058,1.972
3,NTPC,298.20,366.95,334.741,12.466,1.156651e+07,0.033,1.361
4,ONGC,219.84,275.39,243.069,9.327,1.141592e+07,0.059,1.565


In [6]:
long_data = pd.concat([df.assign(symbol=symbol) for symbol, df in stocks.items()], ignore_index=True)

fig = px.line(
    long_data,
    x="date",
    y="close",
    color="symbol",
    facet_row="symbol",
    title="Daily Closing Prices",
    height=1000,
    width=1000,
    labels={"close":"Close (INR)", "date":"Date"},
    render_mode="svg"
)

fig.update_annotations(visible=False)
fig.update_yaxes(matches=None)
fig.show()
fig.write_image("../images/closing-prices.png", width=1000, height=1000, scale=2)

In [7]:
indexed = pd.concat([df.assign(symbol=symbol, indexed_close=100 * df["close"] / df["close"].iloc[0]) for symbol, df in stocks.items()], ignore_index=True)

fig = px.line(
    indexed,
    x="date",
    y="indexed_close",
    color="symbol",
    title="Closing Prices Indexed to 100",
    labels={"date":"Date", "indexed_close":"Indexed close"},
    width=1000,
    height=450,
    render_mode="svg"
)
fig.show()
fig.write_image("../images/closing-prices-indexed.png", width=1000, height=400, scale=2)

In [8]:
for symbol, df in stocks.items():
    fig = go.Figure()
    
    for column in ["close", "MA10", "MA20", "MA50"]:
        fig.add_trace(go.Scatter(x=df["date"], y=df[column], name=column, mode="lines"))
    
    fig.update_layout(
        title=f"{symbol}: Closing Price and Moving Averages",
        xaxis_title="Date",
        yaxis_title="Price (INR)",
        width=1000,
        height=450
    )
    
    fig.show()
    
    fig.write_image(f"../images/{symbol}-closing-price-ma.png", width=1000, height=400, scale=2)

In [9]:
fig = px.bar(
    long_data, 
    x="date", 
    y="volume", 
    color="symbol", 
    facet_row="symbol",
    title="Daily Trading Volume", 
    width=1000, 
    height=1000,
    labels={"volume":"Volume (shares)", "date":"Date"}
)
fig.update_annotations(visible=False)
fig.update_yaxes(matches=None)
fig.show()
fig.write_image("../images/daily-trading-volume.png", width=1000, height=1000, scale=2)

In [10]:
fig = px.line(
    long_data, 
    x="date",
    y="return_pct",
    color="symbol",
    title="Daily Percentage Returns",
    labels={"date":"Date", "return_pct":"Daily return (%)"}, 
    width=1000,
    height=450,
    render_mode="svg"
)
fig.show()
fig.write_image("../images/daily-percentage-returns.png", width=1000, height=400, scale=2)

In [11]:
fig = px.histogram(
    long_data.dropna(subset=["return_pct"]),
    x="return_pct",
    color="symbol",
    nbins=40,
    barmode="overlay",
    opacity=0.5,
    histnorm="probability density",
    title="Distribution of Daily Returns",
    labels={"return_pct":"Daily return (%)"},
    width=1000,
    height=450
)
fig.show()
fig.write_image("../images/distribution-of-daily-returns.png", width=1000, height=400, scale=2)

In [12]:
prices = pd.concat({symbol:df.set_index("date")["close"] for symbol, df in stocks.items()}, axis=1)

returns = prices.pct_change(fill_method=None) * 100

for title, values in [("Closing Price Correlation", prices), ("Daily Return Correlation", returns)]:
    corr = values.dropna().corr()
    
    fig = px.imshow(
        corr, 
        text_auto=".2f", 
        zmin=-1, 
        zmax=1, 
        color_continuous_scale="RdBu_r",  
        title=title, 
        width=900, 
        height=550
    )
    fig.show()
    fig.write_image(f"../images/{'-'.join(title.lower().split())}.png", width=1000, height=400, scale=2)

In [13]:
fig = px.scatter(
    summary, 
    x="daily_volatility_pct", 
    y="mean_daily_return_pct", 
    text="symbol", 
    color="symbol",
    title="Historical Daily Risk Versus Return",
    labels={"daily_volatility_pct":"Daily volatility (%)", "mean_daily_return_pct":"Mean daily return (%)"},
    width=1000, 
    height=450
)
fig.show()
fig.write_image("../images/daily-risk-vs-return.png", width=1000, height=400, scale=2)